In [2]:
# generate_report.py

# generate_report.py
# ---------------------------------------------------------
# This script:
# 1. Runs a single analytical query to identify the
#    top-selling products.
# 2. Exports the query results to a CSV file
#    (top_products.csv).
# 3. Generates a simple text-based summary report
#    (report.txt) with key metrics.
# ---------------------------------------------------------

import sqlite3
import csv
from datetime import datetime
from pathlib import Path


def get_db_path() -> str:
    """Return the path to the SQLite database file."""
    try:
        base_dir = Path(__file__).parent
    except NameError:
        # Fallback if run in a Jupyter environment
        base_dir = Path.cwd()
    return str(base_dir / "data" / "myproject.db")


def export_to_csv(data, filename, headers):
    """Export query results to a CSV file."""
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(headers)
        writer.writerows(data)
    print(f"CSV exported: {filename}")


def generate_summary_report(db_path: str, output_file: str = "report.txt") -> None:
    """Generate a simple text-based summary report."""
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()

        # Total revenue across all orders
        cursor.execute("""
            SELECT SUM(o.quantity * p.price)
            FROM orders o
            JOIN products p ON o.product_id = p.product_id;
        """)
        total_revenue = cursor.fetchone()[0] or 0

        # Total number of orders
        cursor.execute("SELECT COUNT(*) FROM orders;")
        total_orders = cursor.fetchone()[0] or 0

        # Total number of customers
        cursor.execute("SELECT COUNT(*) FROM customers;")
        total_customers = cursor.fetchone()[0] or 0

        # Average order value
        avg_order_value = total_revenue / total_orders if total_orders > 0 else 0

        # Best-selling category by quantity
        cursor.execute("""
            SELECT p.category, SUM(o.quantity) AS total_qty
            FROM orders o
            JOIN products p ON o.product_id = p.product_id
            GROUP BY p.category
            ORDER BY total_qty DESC
            LIMIT 1;
        """)
        best_category_row = cursor.fetchone()
        best_category = best_category_row[0] if best_category_row else None
        best_category_qty = best_category_row[1] if best_category_row else 0

    # Write the text report
    with open(output_file, "w", encoding="utf-8") as f:
        f.write("=== SALES SUMMARY REPORT ===\n")
        f.write(f"Generated on: {datetime.now()}\n\n")
        f.write(f"Total revenue: ${total_revenue:,.2f}\n")
        f.write(f"Total orders: {total_orders}\n")
        f.write(f"Total customers: {total_customers}\n")
        f.write(f"Average order value: ${avg_order_value:,.2f}\n")
        if best_category:
            f.write(
                f"Best-selling category: {best_category} "
                f"(Total quantity sold: {best_category_qty})\n"
            )

    print(f"Text report generated: {output_file}")


def main():
    db_path = get_db_path()

    # 1) Run the main analytical query: top-selling products
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("""
            SELECT p.name,
                   SUM(o.quantity) AS total_quantity,
                   SUM(o.quantity * p.price) AS total_revenue
            FROM orders o
            JOIN products p ON o.product_id = p.product_id
            GROUP BY p.product_id, p.name
            ORDER BY total_quantity DESC
            LIMIT 10;
        """)
        top_products_data = cursor.fetchall()

    # 2) Export to CSV
    export_to_csv(
        top_products_data,
        filename="top_products.csv",
        headers=["product_name", "total_quantity", "total_revenue"]
    )

    # 3) Generate summary report
    generate_summary_report(db_path, output_file="report_task1-3.txt")


if __name__ == "__main__":
    main()

CSV exported: top_products.csv
Text report generated: report_task1-3.txt
